# Repartition vs Coalesce in PySpark

This notebook presents hands-on examples and explanations for `repartition()` and `coalesce()` methods in PySpark DataFrames, highlighting their behavior, performance impact, and use cases. Includes comparison demos with row and partition counts, practical scenarios, and best practices for optimizing shuffle and minimizing unnecessary partitioning in Spark workloads.

## ⚠️ Platform Note — RDD Operations Are NOT Allowed on Serverless Compute

* Databricks **serverless** compute runs **Spark Connect**: your Python code executes in a thin client while Spark runs on the server — RDDs are driver-local objects and **cannot cross the gRPC connection**
* ❌ Not available on this compute: `df.rdd`, `rdd.getNumPartitions()`, `sparkContext.parallelize()`, `spark.sparkContext`, and any `_jvm` / private JVM access
* That is why every commented-out `rdd.getNumPartitions()` line in this notebook is replaced by the Spark Connect-safe pattern:

  `df.select(spark_partition_id()).distinct().count()`

* ✅ The full DataFrame API — including `repartition()` / `coalesce()` — works normally; **only the RDD API is blocked**

# Setup — Paths & Imports

Shared constants used across this notebook:

* `setup_data_dirs(dir_name)` — reusable helper: copies `data/{dir_name}` from the repo into a UC volume and creates its matching output volume (`{dir_name}` → `/Volumes/workspace/default/{dir_name}`, outputs → `/Volumes/workspace/default/{dir_name}-output`)
* `FLIGHT_DATA_DIR` / `OUTPUT_DIR` — dataset and write-output paths, derived from the helper
* `pprint` — pretty-printer for readable `dtypes` and schema output

In [0]:
import shutil
from pathlib import Path

from pprint import pprint

# Reusable setup — Unit Catalog (UC) volumes are the only workspace locations Spark can both
# READ from and WRITE to. This helper copies a dataset directory from the git
# repo into a UC volume and prepares its matching output directory.
REPO_DATA_ROOT = Path("/Workspace/Repos/mayurkedari16@gmail.com/pyspark-databricks/data")
VOLUME_ROOT = Path("/Volumes/workspace/default")


def setup_data_dirs(
    dir_name: str,
    data_volume: str | None = None,
    output_volume: str | None = None,
) -> tuple[str, str]:
    """
    Copy <repo>/data/{dir_name} into a UC volume and create its output directory.

    * data:   {REPO_DATA_ROOT}/{dir_name}  ->  {VOLUME_ROOT}/{data_volume or dir_name}
    * output:                                 {VOLUME_ROOT}/{output_volume or {dir_name}-output}

    Creates the volumes if they don't exist (CREATE VOLUME IF NOT EXISTS) and is
    idempotent — re-running safely overwrites the staged copies.

    Returns (data_dir, output_dir) as strings; data_dir ends in "/"
    for path concatenation (FLIGHT_DATA_DIR + "csv/*.csv").
    """
    src = Path(REPO_DATA_ROOT) / dir_name
    if not src.is_dir():
        raise FileNotFoundError(f"No dataset directory at {src}")

    data_volume = data_volume or dir_name
    output_volume = output_volume or f"{dir_name}-output"

    # A volume must exist in Unity Catalog before its /Volumes path is usable
    spark.sql(f"CREATE VOLUME IF NOT EXISTS workspace.default.`{data_volume}`")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS workspace.default.`{output_volume}`")

    data_dir = Path(VOLUME_ROOT) / data_volume
    output_dir = Path(VOLUME_ROOT) / output_volume

    shutil.copytree(src, data_dir, dirs_exist_ok=True)
    output_dir.mkdir(parents=True, exist_ok=True)

    return data_dir.as_posix() + "/", output_dir.as_posix() + "/"

In [0]:
# Wire up this notebook's dataset using the reusable helper — copies
# data/flight-data from the repo into the flight_data volume (idempotent)
# and points OUTPUT_DIR at the flight_output volume.
FLIGHT_DATA_DIR, OUTPUT_DIR = setup_data_dirs(
    "flight-data",
    data_volume="flight_data",
    output_volume="flight_output",
)

print("Dataset Dir:", FLIGHT_DATA_DIR)
print("Outputs Dir:", OUTPUT_DIR)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, expr
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("repartitioning-and-coalescing").getOrCreate()

flight_df = (
    spark.read.format("csv")
    .option("inferSchema", "true")
    .option("header", "true")
    .load(FLIGHT_DATA_DIR + "csv/*.csv")
)
flight_df.show()

In [0]:
flight_df.count()

In [0]:
# print("Total Partitions: ", flight_df.rdd.getNumPartitions())  # ❌ NOT available on Spark Connect (serverless) — RDD API is driver-local

from pyspark.sql.functions import spark_partition_id

# Count how many distinct partition IDs exist in the active DataFrame
# (Spark Connect-safe alternative to rdd.getNumPartitions())
partition_count = flight_df.select(spark_partition_id()).distinct().count()
print(partition_count)

## Repartition — always shuffles, any direction, balanced

* `repartition(n)` → exactly **n** partitions via a full **shuffle** (expensive but even)
* Works in **both directions** — can increase AND decrease the partition count
* `repartition(n, "col")` → hash-partitions on the column — the same key always lands in the same partition
* `repartitionByRange(n, "col")` → **sorts** by the column into **n contiguous non-overlapping ranges** (boundaries sampled — approximate)

In [0]:
# Repartition the DataFrame into 4 partitions
partitioned_flight_df = flight_df.repartition(4)

(
    partitioned_flight_df
    .withColumn("partition_id", spark_partition_id())
    .groupBy(col("partition_id"))
    .count()
    .orderBy(col("partition_id"))
).show()

In [0]:
# Repartition BY COLUMN — hash-partitions the rows into 300 partitions keyed on "origin_country_name"
# (rows with the same country always land in the SAME partition — hash of the key;
#  empty partitions don't appear in the output, so the visible count can be < 300)
repartition_on_column_df = flight_df.repartition(300, "origin_country_name")

(
    repartition_on_column_df
    .withColumn("partition_id", spark_partition_id())
    .groupBy(col("partition_id"))
    .count()
    .orderBy(col("partition_id"))
).show()

### `repartitionByRange(n, "col")` — range-based partitioning

* **Sorts** the data by the column and splits it into **n contiguous ranges** — one partition per range
* Unlike `repartition(n, "col")` (HASH → same key = same partition), each partition holds a **non-overlapping [min, max] value range**
* ⚠️ Boundaries are **approximate** — Spark samples the data to pick them, so ranges are not exact quantiles
* Also a full **shuffle**, like `repartition()`
* Best for: writing sorted output, range pruning on numeric / date columns

In [0]:
# repartitionByRange(n, "col") — SORTS by the column and splits into n CONTIGUOUS RANGES
# (unlike repartition(n, col) which HASHES the key — here each partition holds a
#  non-overlapping [min, max] range of the column)
# ⚠️ Boundaries are APPROXIMATE — Spark SAMPLES the data to pick them
range_partitioned_df = flight_df.repartitionByRange(4, "count")

# Show each partition's row count AND its [min, max] range — the ranges do not overlap
(
    range_partitioned_df
    .withColumn("partition_id", spark_partition_id())
    .groupBy(col("partition_id"))
    .agg(
        F.count("*").alias("rows"),
        F.min("count").alias("range_min"),
        F.max("count").alias("range_max"),
    )
    .orderBy(col("partition_id"))
).show()

## Coalesce — shrink only, no full shuffle

* `coalesce(n)` → reduces partitions **without** a full shuffle (fast)
* Can only **decrease** (or keep) the partition count — requesting more is a **no-op**
* ⚠️ Merges existing partitions instead of reshuffling → can produce **uneven / skewed** partitions

In [0]:
# Coalesce the DataFrame into 3 partitions - Uneven / Skewed Partitions

coalesce_flight_df = flight_df.coalesce(3)

(
    coalesce_flight_df
    .withColumn("partition_id", spark_partition_id())
    .groupBy(col("partition_id"))
    .count()
    .orderBy(col("partition_id"))
).show()
# print("Total Partitions: ", coalesce_flight_df.rdd.getNumPartitions())



In [0]:
# Try to INCREASE the coalesced partitions (3 -> 8) — VERIFIED RESULT: 6 partitions!
# coalesce() can NEVER grow partitions beyond the upstream count — the optimizer
# COLLAPSES the chained coalesce(3).coalesce(8) into coalesce(8) on the ORIGINAL
# 6-partition flight_df, which is a NO-OP -> result is 6 (not 3, not 8)
recoalesce_to_8_df = coalesce_flight_df.coalesce(8)

# Spark Connect-safe partition count — count the distinct partition IDs
recoalesce_partition_count = (
    recoalesce_to_8_df
    .select(spark_partition_id().alias("partition_id"))
    .distinct()
    .count()
)
print("Original: 6 | coalesce(3): 3 | coalesce(3).coalesce(8):", recoalesce_partition_count)
print("-> NOT 8! The chain collapsed and coalesce(8) no-op'd over the original 6 partitions")
print("-> coalesce() NEVER increases partitions — use repartition(n) to grow (full shuffle)")


In [0]:
# Repartition the DataFrame into 3 partitions - Even / Uniform Sized Partitions

repartition_flight_df = flight_df.repartition(3)

(
    repartition_flight_df
    .withColumn("partition_id", spark_partition_id())
    .groupBy(col("partition_id"))
    .count()
    .orderBy(col("partition_id"))
).show()
# print("Total Partitions: ", repartition_flight_df.rdd.getNumPartitions())


## Why manage partitions? — Spark shuffle

- Efficient partitioning impacts performance in Spark jobs (especially large datasets)
- `repartition(n)` reshuffles to exactly n partitions (adds or reduces, always shuffles)
- `coalesce(n)` reduces the number of partitions without full shuffle; fast when shrinking partitions
- `repartitionByRange(n, "col")` sorts the data into n contiguous value ranges (same full shuffle as repartition)
- Over-partitioning slows down tasks; under-partitioning can create skew or underutilize resources

### Key differences — repartition() vs repartitionByRange() vs coalesce()

| Feature                   | repartition(n)           | repartitionByRange(n, "col")      | coalesce(n)                    |
|---------------------------|--------------------------|-----------------------------------|--------------------------------|
| Shuffle                   | Always (expensive)       | Always (expensive)                | Never (increase = no-op)      |
| Partition count change    | Any (increase/decrease)  | Any (increase/decrease)           | Only decrease (or same)        |
| Layout                    | Hash — balanced rows     | Sorted — contiguous value ranges  | Merges — no reshuffle         |
| Typical use cases         | Wide reshuffles, balanced distribution | Sorted output, range pruning | Efficient shrinking after a wide shuffle |
| Null/empty partition risk | Balanced after shuffle   | Balanced if data is well spread   | May leave uneven partitions    |

### Practical advice

- Use `repartition()` when you need a specific partition count or balanced data after a big shuffle (e.g., wide aggregations, joins).
- Use `coalesce()` after a shuffle if you need to reduce (and only reduce) the number of partitions efficiently — avoids full shuffle.
- Always check partition count before and after processing to avoid bottlenecks or resource waste:
  - Too many partitions can create overhead (especially small files on output)
  - Too few partitions concentrate workload, risking skew and slow performance
- For most output stages, aim for 1 partition per executor core for optimal parallelism.
- Never use `coalesce()` to increase the partition count — it CANNOT increase and never shuffles (verified: `coalesce(8)` after `coalesce(3)` was a no-op returning the original 6 partitions); use `repartition(n)` to grow.

---
Try these cells with larger datasets, inspect partition counts, and time operations for performance awareness.

# 📓 Quick Revision Notes — repartition() vs coalesce() (Traditional Notebook Style)

> Copy these into a hard-copy notebook for offline revision. Short lines, easy to write by hand.

## 1. Setup Notes

* UC volumes (`/Volumes/...`) = the only workspace locations Spark can READ + WRITE
* `setup_data_dirs(dir_name)` → copies repo data into a UC volume + creates its output volume (idempotent)
* ⚠️ Spark Connect (serverless): NO `rdd.getNumPartitions()` — count partitions with `df.select(spark_partition_id()).distinct().count()`

## 2. `repartition(n)`

* Full **shuffle** → expensive, but produces **evenly sized** partitions
* Works in **BOTH directions** — increase AND decrease
* `repartition(n, "col")` → hash by column → same key lands in the same partition
* `repartitionByRange(n, "col")` → **sorts** by the column → n **contiguous non-overlapping ranges** (boundaries **sampled** — approximate)
  * Verified: `repartitionByRange(4, "count")` → 4 non-overlapping ranges, balanced row counts — boundaries VARY between runs (sampling!): [1–17], [18–69], [70–266], [267–370002] on one run, [1–15], [16–62], ... on another

## 3. `coalesce(n)`

* **Shrink only**, avoids full shuffle → fast when reducing
* ⚠️ `n >` upstream count = **NO-OP** — can NEVER grow partitions
  * Verified: `coalesce(3).coalesce(8)` → **6** partitions (the ORIGINAL count!) — the optimizer collapses the chain, the inner coalesce is dropped
* Can produce **skewed / uneven** partitions (merges partitions, no reshuffle)

## 4. Comparison Table

| Feature | `repartition(n)` | `repartitionByRange(n, "col")` | `coalesce(n)` |
| --- | --- | --- | --- |
| Shuffle | always (expensive) | always (expensive) | never (increase = no-op) |
| Direction | increase + decrease | increase + decrease | decrease only |
| Layout | hash — balanced rows | sorted — contiguous value ranges | merge — no reshuffle |
| Partition sizes | even / balanced | even if data spread well | may be uneven / skewed |
| Best for | rebalance after wide op, grow partitions | sorted output, range pruning | efficient shrink before output |

## 5. Practical Rules

* Aim for ~1 partition per executor core for optimal parallelism
* Too many partitions → task overhead + small output files
* Too few → skew + underutilized cores
* Typical flow: wide shuffle (join/groupBy) → `repartition()` to rebalance → `coalesce(1)` for a single-file output

## 6. Quick Recall Box ⭐

* repartition = shuffle, balanced, both directions
* coalesce = no shuffle, shrink only, may skew
* `coalesce(8)` after `coalesce(3)` → **6** = the ORIGINAL count (chain collapsed, no-op — verified)
* Chained coalesces are NOT sequential — the optimizer may drop the inner one
* `repartition(300, col)` = hash partitions on the column (same key → same partition)
* `repartitionByRange(n, col)` = sorted contiguous value ranges — vs hash: same value-range, not same key
* Both repartition variants do a full shuffle; coalesce does not
* Spark Connect: count partitions via `spark_partition_id()`, never `rdd`
* To grow partitions → ALWAYS `repartition()`, never `coalesce()`